# Brain Tumor Detection — YOLOv8n Training on Google Colab

**Pipeline steps covered here:** Step 3 (setup) · Step 4 (training) · Step 6 (export)

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 or better)
2. Upload `brain_tumor_dataset.zip` to your Drive, OR paste the File ID from `step2_upload.py` into `GDRIVE_ZIP_ID` below
3. Run all cells top-to-bottom

---
## Cell 1 — Install Dependencies

In [ ]:
# Install Ultralytics (brings in PyTorch, OpenCV, etc.)
# The %pip magic ensures the package is installed into the active kernel
%pip install ultralytics --quiet

# Verify GPU is available
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — go to Runtime → Change runtime type → GPU")

---
## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Convenience alias for the Drive root
DRIVE_ROOT = '/content/drive/MyDrive'

# All training output (weights, logs, plots) will land here
PROJECT_DIR = f'{DRIVE_ROOT}/BrainTumor'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Drive mounted. Project folder: {PROJECT_DIR}")

---
## Cell 3 — Load Dataset from Drive

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ── Option A: you uploaded the ZIP manually to Drive ─────────────────────────
# Set this to the path where you placed the ZIP in your Drive
ZIP_ON_DRIVE = f'{DRIVE_ROOT}/brain_tumor_dataset.zip'

# ── Option B: use the File ID printed by step2_upload.py ─────────────────────
# Leave as None if you're using Option A, otherwise paste your file ID here
GDRIVE_ZIP_ID = None   # e.g.  '1AbCdEfGhIjKlMnOpQrStUvWxYz'

DATASET_DIR = '/content/dataset'

if GDRIVE_ZIP_ID:
    # Download from Drive by File ID using gdown
    !pip install gdown --quiet
    import gdown
    zip_local = '/content/brain_tumor_dataset.zip'
    gdown.download(id=GDRIVE_ZIP_ID, output=zip_local, quiet=False)
    zip_path = zip_local
else:
    zip_path = ZIP_ON_DRIVE

# Extract
if os.path.exists(DATASET_DIR):
    shutil.rmtree(DATASET_DIR)   # clean any previous extraction

print(f"Extracting {zip_path} → {DATASET_DIR} ...")
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall('/content')

# The ZIP contains a top-level 'brain_tumor_dataset/' folder
extracted = '/content/brain_tumor_dataset'
if os.path.exists(extracted):
    os.rename(extracted, DATASET_DIR)

print("Dataset structure:")
for split in ['train', 'valid', 'test']:
    imgs  = list(Path(DATASET_DIR).glob(f'{split}/images/*'))
    lbls  = list(Path(DATASET_DIR).glob(f'{split}/labels/*'))
    print(f"  {split:6s}: {len(imgs):4d} images,  {len(lbls):4d} labels")

---
## Cell 4 — Configure data.yaml

In [ ]:
import yaml

DATA_YAML = f'{DATASET_DIR}/data.yaml'

# Update the 'path' key to point to the extracted location on Colab
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

cfg['path'] = DATASET_DIR          # absolute path so YOLOv8 can resolve splits

with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print("data.yaml (updated):")
print("-" * 40)
with open(DATA_YAML) as f:
    print(f.read())

---
## Cell 5 — Initialise the YOLOv8n Model

In [ ]:
from ultralytics import YOLO

# yolov8n.pt = nano (lightest, ideal for mobile export)
# Pre-trained on COCO — transfer-learning saves ~30% of epochs vs random init
model = YOLO('yolov8n.pt')

print(f"Model  : YOLOv8n")
print(f"Params : {sum(p.numel() for p in model.model.parameters()):,}")

---
## Cell 6 — Train

Training writes **directly to Google Drive** via the `project` argument, so:
- `best.pt`   → saved whenever validation mAP improves
- `last.pt`   → overwritten at the end of every epoch
- Epoch snapshots → saved every `save_period` epochs

If Colab disconnects and you restart, set `resume=True` to pick up from `last.pt`.

In [ ]:
# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS      = 100
IMG_SIZE    = 640    # native YOLOv8 input; matches our dataset
BATCH_SIZE  = 16     # fits T4 (16 GB) comfortably; lower to 8 if OOM
PATIENCE    = 20     # early-stop if mAP doesn't improve for 20 epochs
SAVE_PERIOD = 10     # save epochN.pt snapshot every 10 epochs

# ── Train ────────────────────────────────────────────────────────────────────
results = model.train(
    data        = DATA_YAML,
    epochs      = EPOCHS,
    imgsz       = IMG_SIZE,
    batch       = BATCH_SIZE,
    patience    = PATIENCE,
    save_period = SAVE_PERIOD,
    # Writes ALL outputs (weights/, plots/, results.csv) straight to Drive
    project     = PROJECT_DIR,
    name        = 'train',
    exist_ok    = True,     # don't crash if the folder already exists
    resume      = False,    # set True to resume from last.pt after disconnection
    # Augmentation (good defaults for medical images)
    degrees     = 10.0,     # random rotation ±10°
    flipud      = 0.3,      # vertical flip probability
    fliplr      = 0.5,      # horizontal flip probability
    hsv_h       = 0.015,
    hsv_s       = 0.4,
    hsv_v       = 0.4,
    device      = 0,        # use GPU 0; set to 'cpu' if no GPU
    workers     = 4,
    verbose     = True,
)

WEIGHTS_DIR = f'{PROJECT_DIR}/train/weights'
print(f"\nTraining complete!")
print(f"  best.pt → {WEIGHTS_DIR}/best.pt")
print(f"  last.pt → {WEIGHTS_DIR}/last.pt")

---
## Cell 7 — Plot Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_path = f'{PROJECT_DIR}/train/results.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()   # strip any accidental whitespace

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('YOLOv8n Training Curves — Brain Tumor Detection', fontsize=14, fontweight='bold')

plot_pairs = [
    ('train/box_loss',  'val/box_loss',  'Box Loss'),
    ('train/cls_loss',  'val/cls_loss',  'Classification Loss'),
    ('train/dfl_loss',  'val/dfl_loss',  'DFL Loss'),
    ('metrics/mAP50(B)',    None,            'mAP@50'),
    ('metrics/mAP50-95(B)', None,            'mAP@50-95'),
    ('metrics/precision(B)', 'metrics/recall(B)', 'Precision / Recall'),
]

for ax, (train_col, val_col, title) in zip(axes.flatten(), plot_pairs):
    epochs = df['epoch']
    if train_col in df.columns:
        label = 'val' if val_col is None else 'train'
        ax.plot(epochs, df[train_col], label=label, linewidth=2)
    if val_col and val_col in df.columns:
        ax.plot(epochs, df[val_col], label='val', linewidth=2, linestyle='--')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved training_curves.png to {PROJECT_DIR}")

---
## Cell 8 — Evaluate on Test Set

In [ ]:
from ultralytics import YOLO

WEIGHTS_DIR = f'{PROJECT_DIR}/train/weights'
best_model  = YOLO(f'{WEIGHTS_DIR}/best.pt')

test_results = best_model.val(
    data  = DATA_YAML,
    split = 'test',
    imgsz = 640,
    conf  = 0.25,
    iou   = 0.5,
    verbose = True,
)

print("\n── Test Set Metrics ─────────────────────────────────")
print(f"  mAP@50     : {test_results.box.map50:.4f}")
print(f"  mAP@50-95  : {test_results.box.map:.4f}")
print(f"  Precision  : {test_results.box.mp:.4f}")
print(f"  Recall     : {test_results.box.mr:.4f}")

CLASS_NAMES = ['glioma', 'meningioma', 'pituitary']
print("\n── Per-Class AP@50 ──────────────────────────────────")
for i, name in enumerate(CLASS_NAMES):
    ap = test_results.box.ap50[i] if i < len(test_results.box.ap50) else 0
    print(f"  {name:<14}: {ap:.4f}")

---
## Cell 9 — Visualise Sample Predictions

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from PIL import Image

CLASS_NAMES  = ['glioma', 'meningioma', 'pituitary']
CLASS_COLORS = [(0.96, 0.26, 0.21), (0.13, 0.59, 0.95), (0.30, 0.69, 0.31)]

test_img_dir = Path(f'{DATASET_DIR}/test/images')
samples      = random.sample(list(test_img_dir.glob('*.jpg')), 8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Predictions — YOLOv8n Brain Tumor', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes.flatten(), samples):
    img = np.array(Image.open(img_path).convert('RGB'))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(img_path.stem[:22], fontsize=7)

    results = best_model.predict(str(img_path), conf=0.25, verbose=False)[0]
    h, w    = img.shape[:2]

    for box in results.boxes:
        cid       = int(box.cls.item())
        conf_val  = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        color = CLASS_COLORS[cid] if cid < len(CLASS_COLORS) else (1, 1, 0)
        rect  = mpatches.FancyBboxPatch(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none',
            boxstyle=mpatches.BoxStyle('Square', pad=0),
        )
        ax.add_patch(rect)
        label = f"{CLASS_NAMES[cid] if cid < len(CLASS_NAMES) else cid} {conf_val:.2f}"
        ax.text(x1, max(y1 - 4, 0), label, fontsize=7, color='white', fontweight='bold',
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 10 — Step 6: Export Models

Three export formats:
| Format | Use case | File |
|--------|----------|------|
| TFLite (int8) | Flutter mobile app | `best_saved_model/best_int8.tflite` |
| ONNX | Python backend / FastAPI | `best.onnx` |
| SavedModel | TF Serving / Vertex AI | `best_saved_model/` |

In [ ]:
import shutil, os
from ultralytics import YOLO
from pathlib import Path

WEIGHTS_DIR  = f'{PROJECT_DIR}/train/weights'
EXPORTS_DIR  = f'{PROJECT_DIR}/exports'
os.makedirs(EXPORTS_DIR, exist_ok=True)

best_model = YOLO(f'{WEIGHTS_DIR}/best.pt')

# ── 1. TFLite (int8 quantised) — for Flutter ─────────────────────────────────
print("Exporting → TFLite (int8) ...")
tflite_export = best_model.export(
    format = 'tflite',
    imgsz  = 640,
    int8   = True,    # quantise weights to 8-bit → ~4× smaller, faster on mobile
)
# The export() returns the path to the generated SavedModel folder
# The .tflite file is inside it
tflite_folder = Path(tflite_export)
tflite_files  = list(tflite_folder.glob('*.tflite')) if tflite_folder.is_dir() else [Path(str(tflite_export))]
for f in tflite_files:
    shutil.copy(f, f'{EXPORTS_DIR}/{f.name}')
    print(f"  ✓ {f.name}  →  {EXPORTS_DIR}/")

# ── 2. ONNX — for Python backend ─────────────────────────────────────────────
print("\nExporting → ONNX ...")
onnx_path = best_model.export(
    format  = 'onnx',
    imgsz   = 640,
    dynamic = False,  # fixed input shape → faster inference in production
    opset   = 17,
)
shutil.copy(onnx_path, f'{EXPORTS_DIR}/best.onnx')
print(f"  ✓ best.onnx  →  {EXPORTS_DIR}/")

# ── 3. TF SavedModel — for TF Serving / Vertex AI ────────────────────────────
print("\nExporting → TF SavedModel ...")
saved_model_path = best_model.export(
    format = 'saved_model',
    imgsz  = 640,
)
dst = f'{EXPORTS_DIR}/saved_model'
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(saved_model_path, dst)
print(f"  ✓ saved_model/  →  {EXPORTS_DIR}/")

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "=" * 50)
print("  Exports saved to Drive:")
for f in sorted(Path(EXPORTS_DIR).rglob('*')):
    if f.is_file():
        size_mb = f.stat().st_size / 1_048_576
        print(f"  {str(f.relative_to(EXPORTS_DIR)):<40} {size_mb:6.1f} MB")
print("=" * 50)

---
## Cell 11 — Flutter Integration Notes

After downloading `best_int8.tflite` from Drive:

1. Place the file in your Flutter project at `assets/best_int8.tflite`
2. Add to `pubspec.yaml`:
   ```yaml
   flutter:
     assets:
       - assets/best_int8.tflite
   ```
3. Add dependency:
   ```yaml
   dependencies:
     tflite_flutter: ^0.10.4
   ```
4. Input tensor shape: `[1, 640, 640, 3]`  (float32, values 0-1)
5. Output tensor: `[1, 7, 8400]`  — 7 = (4 bbox coords + 3 class scores), 8400 anchors
   Apply NMS post-processing to get final detections.

A minimal Flutter inference snippet is in `mobile/README_model.md`.

---
## Appendix — Resume Training After Disconnection

If Colab disconnects mid-training, re-run cells 1-5, then run this cell instead of Cell 6:

In [ ]:
# Resume from last.pt — only run this if training was interrupted
from ultralytics import YOLO

WEIGHTS_DIR = f'{PROJECT_DIR}/train/weights'
last_pt     = f'{WEIGHTS_DIR}/last.pt'

resume_model = YOLO(last_pt)
resume_model.train(resume=True)  # picks up epoch count and optimizer state automatically